# 1. Introduction

This notebook presents a final experimental study on dimensionality reduction through filter-based feature selection and a manual multi-class LDA classifier.

We compare five from-scratch ranking methods:
- Pearson correlation
- Spearman rank correlation
- Chi-Square statistic
- Gini impurity reduction
- Information Gain

All model evaluation is performed with leakage-safe cross-validation: feature ranking is computed only on each training fold before top-k selection and LDA fitting.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    confusion_matrix,
)

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')


# 2. Dataset Description

We use the Wine dataset (178 samples, 13 continuous physicochemical features, 3 classes).


In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')

print(f'Samples: {X.shape[0]}')
print(f'Features: {X.shape[1]}')
print('Class counts:')
print(y.value_counts().sort_index())
X.head()


# 3. Methodology

## Pearson Correlation
For feature $x_j$ and target $y$, we compute absolute Pearson correlation:
$$
r_{x_j,y} = rac{	ext{cov}(x_j,y)}{\sigma_{x_j}\sigma_y}
$$

## Spearman Rank Correlation
Each variable is transformed to ranks first, then Pearson is applied to ranked values.

## Chi-Square
Each continuous feature is discretized into equal-frequency bins. For each feature, we build a contingency table feature-bin vs class and compute:
$$
\chi^2 = \sum_{i,j}rac{(O_{ij} - E_{ij})^2}{E_{ij}}
$$

## Gini
For a discretized feature, weighted post-split impurity is computed with Gini:
$$
Gini(S)=1-\sum_c p(c)^2
$$
The feature score is impurity reduction.

## Information Gain
For a discretized feature:
$$
IG(S, X_j) = H(S) - \sum_v p(v)H(S\mid X_j=v)
$$
where $H$ is entropy.


In [ ]:
def _safe_std(v):
    return np.std(v, ddof=0) + 1e-12


def _manual_pearson(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a_center = a - a.mean()
    b_center = b - b.mean()
    cov = np.mean(a_center * b_center)
    return cov / (_safe_std(a) * _safe_std(b))


def _rank_average(arr):
    arr = np.asarray(arr)
    order = np.argsort(arr, kind='mergesort')
    ranks = np.empty(len(arr), dtype=float)
    i = 0
    while i < len(arr):
        j = i
        while j + 1 < len(arr) and arr[order[j + 1]] == arr[order[i]]:
            j += 1
        avg_rank = (i + j + 2) / 2.0
        ranks[order[i:j + 1]] = avg_rank
        i = j + 1
    return ranks


def _discretize_equal_freq(values, bins=5):
    values = np.asarray(values, dtype=float)
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.quantile(values, quantiles)
    edges = np.unique(edges)
    if len(edges) <= 2:
        return np.zeros(len(values), dtype=int)
    return np.digitize(values, edges[1:-1], right=True)


def compute_pearson_scores(X_df, y_vec):
    return {col: abs(_manual_pearson(X_df[col].values, y_vec)) for col in X_df.columns}


def compute_spearman_scores(X_df, y_vec):
    y_rank = _rank_average(y_vec)
    scores = {}
    for col in X_df.columns:
        x_rank = _rank_average(X_df[col].values)
        scores[col] = abs(_manual_pearson(x_rank, y_rank))
    return scores


def compute_chi_square_scores(X_df, y_vec, bins=5):
    y_arr = np.asarray(y_vec)
    classes = np.unique(y_arr)
    scores = {}
    for col in X_df.columns:
        x_bin = _discretize_equal_freq(X_df[col].values, bins=bins)
        n_bins = int(x_bin.max()) + 1
        obs = np.zeros((n_bins, len(classes)), dtype=float)
        for i, b in enumerate(x_bin):
            c_idx = np.where(classes == y_arr[i])[0][0]
            obs[b, c_idx] += 1
        row_sum = obs.sum(axis=1, keepdims=True)
        col_sum = obs.sum(axis=0, keepdims=True)
        total = obs.sum()
        exp = (row_sum @ col_sum) / (total + 1e-12)
        chi = np.sum((obs - exp) ** 2 / (exp + 1e-12))
        scores[col] = chi
    return scores


def _entropy_from_counts(counts):
    p = counts / (counts.sum() + 1e-12)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


def compute_information_gain_scores(X_df, y_vec, bins=5):
    y_arr = np.asarray(y_vec)
    classes, class_counts = np.unique(y_arr, return_counts=True)
    base_entropy = _entropy_from_counts(class_counts)
    scores = {}
    for col in X_df.columns:
        x_bin = _discretize_equal_freq(X_df[col].values, bins=bins)
        n_bins = int(x_bin.max()) + 1
        cond_entropy = 0.0
        for b in range(n_bins):
            idx = x_bin == b
            if idx.sum() == 0:
                continue
            _, counts = np.unique(y_arr[idx], return_counts=True)
            cond_entropy += (idx.sum() / len(y_arr)) * _entropy_from_counts(counts)
        scores[col] = base_entropy - cond_entropy
    return scores


def compute_gini_scores(X_df, y_vec, bins=5):
    y_arr = np.asarray(y_vec)
    _, class_counts = np.unique(y_arr, return_counts=True)
    base_probs = class_counts / class_counts.sum()
    base_gini = 1.0 - np.sum(base_probs ** 2)
    scores = {}
    for col in X_df.columns:
        x_bin = _discretize_equal_freq(X_df[col].values, bins=bins)
        n_bins = int(x_bin.max()) + 1
        weighted_gini = 0.0
        for b in range(n_bins):
            idx = x_bin == b
            if idx.sum() == 0:
                continue
            _, counts = np.unique(y_arr[idx], return_counts=True)
            probs = counts / counts.sum()
            gini_b = 1.0 - np.sum(probs ** 2)
            weighted_gini += (idx.sum() / len(y_arr)) * gini_b
        scores[col] = base_gini - weighted_gini
    return scores


RANKERS = {
    'Pearson': lambda Xf, yf, bins: compute_pearson_scores(Xf, yf),
    'Spearman': lambda Xf, yf, bins: compute_spearman_scores(Xf, yf),
    'Chi-Square': lambda Xf, yf, bins: compute_chi_square_scores(Xf, yf, bins=bins),
    'Gini': lambda Xf, yf, bins: compute_gini_scores(Xf, yf, bins=bins),
    'Information Gain': lambda Xf, yf, bins: compute_information_gain_scores(Xf, yf, bins=bins),
}


# 4. Feature Ranking

We compute full-dataset rankings once for interpretation plots only. Final performance metrics are reported from leakage-safe CV.


In [ ]:
def sort_scores(score_dict):
    return pd.Series(score_dict).sort_values(ascending=False)

baseline_bins = 5
full_rankings = {
    method: sort_scores(rank_fn(X, y.values, baseline_bins))
    for method, rank_fn in RANKERS.items()
}

ranking_table = pd.DataFrame(full_rankings)
ranking_table.head(10)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
axes = axes.flatten()
for i, (method, ranking) in enumerate(full_rankings.items()):
    ax = axes[i]
    ranking.sort_values(ascending=True).tail(10).plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'{method} Top-10 Feature Scores')
    ax.set_xlabel('Score')
    ax.set_ylabel('Feature')
axes[-1].axis('off')
plt.show()


# 5. Top-k Feature Selection

We evaluate $k \in [2, 4, 6, 8, 10, 13]$.


# 6. Multi-Class LDA

A manual LDA classifier is implemented by solving the generalized eigenproblem of within-class and between-class scatter matrices. Prediction is done by nearest class centroid in projected space.


In [ ]:
class MultiClassLDA:
    def __init__(self, n_components=None, reg=1e-6):
        self.n_components = n_components
        self.reg = reg
        self.W = None
        self.classes_ = None
        self.centroids_ = None

    def fit(self, X_mat, y_vec):
        X_mat = np.asarray(X_mat, dtype=float)
        y_vec = np.asarray(y_vec)
        n_features = X_mat.shape[1]
        self.classes_ = np.unique(y_vec)
        mean_all = X_mat.mean(axis=0)

        Sw = np.zeros((n_features, n_features))
        Sb = np.zeros((n_features, n_features))

        for c in self.classes_:
            Xc = X_mat[y_vec == c]
            mean_c = Xc.mean(axis=0)
            centered = Xc - mean_c
            Sw += centered.T @ centered
            mean_diff = (mean_c - mean_all).reshape(-1, 1)
            Sb += Xc.shape[0] * (mean_diff @ mean_diff.T)

        Sw += self.reg * np.eye(n_features)
        A = np.linalg.pinv(Sw) @ Sb
        eigvals, eigvecs = np.linalg.eig(A)
        idx = np.argsort(eigvals.real)[::-1]
        eigvecs = eigvecs[:, idx].real

        max_components = len(self.classes_) - 1
        if self.n_components is None:
            self.n_components = max_components
        self.n_components = min(self.n_components, max_components, n_features)

        self.W = eigvecs[:, :self.n_components]

        Z = X_mat @ self.W
        self.centroids_ = {
            c: Z[y_vec == c].mean(axis=0)
            for c in self.classes_
        }
        return self

    def predict(self, X_mat):
        Z = np.asarray(X_mat, dtype=float) @ self.W
        preds = []
        for z in Z:
            dists = {c: np.linalg.norm(z - mu) for c, mu in self.centroids_.items()}
            preds.append(min(dists, key=dists.get))
        return np.array(preds)


# 7. Experimental Setup

For each CV fold:
1. Rank features on training split only.
2. Select top-k features.
3. Train manual LDA on selected training features.
4. Evaluate on validation split.

This procedure prevents feature-selection leakage.


In [ ]:
def evaluate_method_cv(X_df, y_series, method, k, bins=5, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_metrics = []
    aggregate_cm = np.zeros((y_series.nunique(), y_series.nunique()), dtype=int)
    fold_selected = []
    fold_rankings = []

    for tr_idx, va_idx in skf.split(X_df, y_series):
        X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
        y_tr, y_va = y_series.iloc[tr_idx], y_series.iloc[va_idx]

        scores = RANKERS[method](X_tr, y_tr.values, bins)
        ranking = pd.Series(scores).sort_values(ascending=False)
        selected = list(ranking.index[:k])

        model = MultiClassLDA(n_components=min(2, k))
        model.fit(X_tr[selected].values, y_tr.values)
        y_pred = model.predict(X_va[selected].values)

        acc = accuracy_score(y_va, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_va, y_pred, average='macro', zero_division=0
        )
        bacc = balanced_accuracy_score(y_va, y_pred)
        cm = confusion_matrix(y_va, y_pred, labels=sorted(y_series.unique()))
        aggregate_cm += cm

        fold_metrics.append({
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'balanced_accuracy': bacc,
        })
        fold_selected.append(set(selected))
        fold_rankings.append(list(ranking.index))

    metric_df = pd.DataFrame(fold_metrics)
    summary = metric_df.mean().to_dict()
    summary_std = metric_df.std(ddof=0).add_suffix('_std').to_dict()
    return summary | summary_std, aggregate_cm, fold_selected, fold_rankings

k_values = [2, 4, 6, 8, 10, 13]
all_results = []
confusion_store = {}
selection_store = {}
ranking_store = {}

for method in RANKERS:
    for k in k_values:
        summary, cm, fold_selected, fold_rankings = evaluate_method_cv(
            X, y, method=method, k=k, bins=baseline_bins, n_splits=5, seed=SEED
        )
        row = {'method': method, 'k': k} | summary
        all_results.append(row)
        confusion_store[(method, k)] = cm
        selection_store[(method, k)] = fold_selected
        ranking_store[(method, k)] = fold_rankings

results_df = pd.DataFrame(all_results).sort_values(['method', 'k']).reset_index(drop=True)
results_df[['method', 'k', 'accuracy', 'precision', 'recall', 'f1', 'balanced_accuracy']]


# 8. Results and Comparisons

The table below reports macro precision, macro recall, macro F1, balanced accuracy, and accuracy for every method and k.


In [ ]:
results_view = results_df[['method', 'k', 'accuracy', 'precision', 'recall', 'f1', 'balanced_accuracy']]
results_view


In [ ]:
plt.figure(figsize=(10, 6))
for method in RANKERS:
    subset = results_df[results_df['method'] == method]
    plt.plot(subset['k'], subset['accuracy'], marker='o', label=method)
plt.title('Accuracy vs Number of Selected Features (k)')
plt.xlabel('k (top-k selected features)')
plt.ylabel('Cross-validated accuracy')
plt.legend(title='Method')
plt.ylim(0.6, 1.01)
plt.show()


In [ ]:
best_idx = results_df['accuracy'].idxmax()
best_method = results_df.loc[best_idx, 'method']
best_k = int(results_df.loc[best_idx, 'k'])

best_cm = confusion_store[(best_method, best_k)]
plt.figure(figsize=(6, 5))
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Aggregated Confusion Matrix: {best_method}, k={best_k}')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.show()

print(f'Best setting by accuracy: method={best_method}, k={best_k}')
print(results_df.loc[best_idx, ['accuracy', 'precision', 'recall', 'f1', 'balanced_accuracy']])


# 9. Stability Analysis

We evaluate stability in two ways:
- Jaccard similarity between fold-wise top-k feature sets.
- Spearman correlation between full ranking lists across folds.


In [ ]:
def jaccard_similarity(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0


def ranking_to_positions(ranking):
    return {feat: i for i, feat in enumerate(ranking)}


def spearman_between_rankings(rank_a, rank_b):
    pos_a = ranking_to_positions(rank_a)
    pos_b = ranking_to_positions(rank_b)
    feats = rank_a
    a = np.array([pos_a[f] for f in feats], dtype=float)
    b = np.array([pos_b[f] for f in feats], dtype=float)
    return _manual_pearson(_rank_average(a), _rank_average(b))


stability_rows = []
for method in RANKERS:
    for k in k_values:
        sel_folds = selection_store[(method, k)]
        rank_folds = ranking_store[(method, k)]

        jacc_vals = []
        spear_vals = []
        for i in range(len(sel_folds)):
            for j in range(i + 1, len(sel_folds)):
                jacc_vals.append(jaccard_similarity(sel_folds[i], sel_folds[j]))
                spear_vals.append(spearman_between_rankings(rank_folds[i], rank_folds[j]))

        stability_rows.append({
            'method': method,
            'k': k,
            'mean_jaccard': float(np.mean(jacc_vals)),
            'mean_spearman_rank_corr': float(np.mean(spear_vals)),
        })

stability_df = pd.DataFrame(stability_rows)
stability_df.head()


In [ ]:
pivot_j = stability_df.pivot(index='method', columns='k', values='mean_jaccard')
pivot_s = stability_df.pivot(index='method', columns='k', values='mean_spearman_rank_corr')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.heatmap(pivot_j, annot=True, cmap='YlGnBu', vmin=0, vmax=1, ax=axes[0])
axes[0].set_title('Stability Heatmap: Mean Jaccard (Top-k Sets)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Method')

sns.heatmap(pivot_s, annot=True, cmap='magma', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Stability Heatmap: Mean Spearman (Ranking Lists)')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Method')
plt.show()


# 10. Sensitivity Analysis

To study discretization sensitivity, we vary bin counts for methods that require discretization.


In [ ]:
bins_list = [3, 5, 7, 10]
discrete_methods = ['Chi-Square', 'Gini', 'Information Gain']

sensitivity_rows = []
for method in discrete_methods:
    for bins in bins_list:
        ranking = pd.Series(RANKERS[method](X, y.values, bins)).sort_values(ascending=False)
        for rank_idx, feat in enumerate(ranking.index, start=1):
            sensitivity_rows.append({
                'method': method,
                'bins': bins,
                'feature': feat,
                'rank': rank_idx,
            })

sensitivity_df = pd.DataFrame(sensitivity_rows)

for method in discrete_methods:
    plt.figure(figsize=(10, 5))
    top_features = (
        sensitivity_df[(sensitivity_df['method'] == method) & (sensitivity_df['bins'] == 5)]
        .nsmallest(8, 'rank')['feature']
        .tolist()
    )
    plot_df = sensitivity_df[(sensitivity_df['method'] == method) & (sensitivity_df['feature'].isin(top_features))]
    sns.lineplot(data=plot_df, x='bins', y='rank', hue='feature', marker='o')
    plt.gca().invert_yaxis()
    plt.title(f'Ranking Sensitivity to Bin Count: {method}')
    plt.xlabel('Number of bins')
    plt.ylabel('Feature rank (1 = best)')
    plt.legend(title='Feature', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.show()


# 11. Conclusion

This finalized pipeline provides a leakage-safe and reproducible comparison of filter feature-selection methods for multi-class LDA.

Main outcomes:
- Manual implementations were used for all five ranking methods.
- Performance was compared consistently across multiple k values.
- Stability and discretization sensitivity analyses were included to assess robustness.

The notebook is organized as an academic experiment and ready for submission.
